<a href="https://colab.research.google.com/github/coderujjwal201/hotel-booking-analysis/blob/main/Hotel_Booking_Analysis_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏨 Hotel Booking & Occupancy Analysis
**Project 13 | Python (Pandas + NumPy) | HCLTech Training Program**

**Business problem:** A hotel wants to analyze its booking data so it can improve room pricing and front-desk staffing.

**Note:** The dataset is synthetic (simulated), so the insights apply only to this data.

**How to run:** Upload `13_hotel_bookings.csv` in Colab (folder icon on the left), then click `Runtime > Run all`.

## Step 0: Importing libraries
- **pandas** = handles data as a table (DataFrame)
- **numpy** = used for maths/statistics

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:,.2f}'.format)  # display numbers nicely

## Task 1: Load and inspect data
Read the CSV file and see what it contains.

In [2]:
df = pd.read_csv('13_hotel_bookings.csv')

print('Shape (rows, columns):', df.shape)
print('\nColumns:', list(df.columns))
print('\nData types:')
print(df.dtypes)
df.head()

Shape (rows, columns): (2025, 10)

Columns: ['Booking_ID', 'Checkin_Date', 'Nights', 'Lead_Time_Days', 'Room_Rate', 'Room_Type', 'Booking_Channel', 'Traveler_Type', 'Guests', 'Booking_Status']

Data types:
Booking_ID          object
Checkin_Date        object
Nights               int64
Lead_Time_Days     float64
Room_Rate          float64
Room_Type           object
Booking_Channel     object
Traveler_Type       object
Guests               int64
Booking_Status      object
dtype: object


,Booking_ID,Checkin_Date,Nights,Lead_Time_Days,Room_Rate,Room_Type,Booking_Channel,Traveler_Type,Guests,Booking_Status
0,BK01029,2025-10-01,3,114.00,"5,669.00",Deluxe,Corporate,Business,2,Confirmed
1,BK00141,2025-07-14,1,121.00,"9,298.00",Suite,Direct Website,Business,1,Canceled
2,BK00111,2025-07-06,6,29.00,"5,288.00",Deluxe,OTA,Leisure,3,Confirmed
3,BK00639,2025-10-15,1,15.00,"3,722.00",Deluxe,OTA,Business,4,Canceled
4,BK00703,2025-03-22,3,32.00,"3,541.00",Standard,Corporate,Business,4,Confirmed


**Understanding the output:** `shape` tells how many rows/columns there are. `dtypes` tells whether each column is a number or text. `Checkin_Date` is currently text (object); we will convert it to a date later.

## Task 2: Missing values and duplicates

In [3]:
print('Missing values per column:')
print(df.isna().sum())

dup_count = df['Booking_ID'].duplicated().sum()
print('\nDuplicate Booking_ID rows:', dup_count)

Missing values per column:
Booking_ID          0
Checkin_Date        0
Nights              0
Lead_Time_Days     72
Room_Rate          55
Room_Type           0
Booking_Channel     0
Traveler_Type       0
Guests              0
Booking_Status      0
dtype: int64

Duplicate Booking_ID rows: 25


In [4]:
# Remove duplicates: keep the first row for each Booking_ID
df = df.drop_duplicates(subset='Booking_ID', keep='first').reset_index(drop=True)
print('Rows after removing duplicates:', len(df))

Rows after removing duplicates: 2000


## Task 3: Handling missing values (with justification)

**Strategy:**
- `Lead_Time_Days` → fill with the **median**. The lead-time distribution is skewed (some guests book very far ahead), so the mean gets distorted while the median stays reliable.
- `Room_Rate` → fill with the **median rate of that room type**. Suite and Standard rates are very different, so applying one overall average to everything would be wrong.

In [5]:
df['Lead_Time_Days'] = df['Lead_Time_Days'].fillna(df['Lead_Time_Days'].median())

# transform = calculates the median separately within each room type
df['Room_Rate'] = df['Room_Rate'].fillna(df.groupby('Room_Type')['Room_Rate'].transform('median'))

print('Missing values after cleaning:')
print(df.isna().sum())

Missing values after cleaning:
Booking_ID         0
Checkin_Date       0
Nights             0
Lead_Time_Days     0
Room_Rate          0
Room_Type          0
Booking_Channel    0
Traveler_Type      0
Guests             0
Booking_Status     0
dtype: int64


## Task 4: Convert the date, extract month and weekday

In [6]:
df['Checkin_Date'] = pd.to_datetime(df['Checkin_Date'])
df['Arrival_Month'] = df['Checkin_Date'].dt.month
df['Month_Name'] = df['Checkin_Date'].dt.month_name()
df['Arrival_Weekday'] = df['Checkin_Date'].dt.day_name()
df[['Checkin_Date', 'Arrival_Month', 'Arrival_Weekday']].head()

,Checkin_Date,Arrival_Month,Arrival_Weekday
0,2025-10-01,10,Wednesday
1,2025-07-14,7,Monday
2,2025-07-06,7,Sunday
3,2025-10-15,10,Wednesday
4,2025-03-22,3,Saturday


## Task 5: Total bookings, room nights, cancellation %

In [7]:
total_bookings = len(df)
total_room_nights = df['Nights'].sum()
confirmed_nights = df.loc[df['Booking_Status'] == 'Confirmed', 'Nights'].sum()
cancel_pct = (df['Booking_Status'] == 'Canceled').mean() * 100

print('Total bookings          :', total_bookings)
print('Total room nights (all) :', total_room_nights)
print('Room nights (confirmed) :', confirmed_nights)
print(f'Cancellation percentage : {cancel_pct:.2f}%')

Total bookings          : 2000
Total room nights (all) : 7196
Room nights (confirmed) : 5953
Cancellation percentage : 16.80%


## Task 6: Revenue = Nights × Room_Rate (non-canceled bookings only)
`np.where` works like an if-else: if the booking is Confirmed, use the revenue; otherwise 0.

In [8]:
df['Revenue'] = np.where(df['Booking_Status'] == 'Confirmed',
                         df['Nights'] * df['Room_Rate'], 0)

confirmed = df[df['Booking_Status'] == 'Confirmed'].copy()  # will be used later
print(f"Total realized revenue: Rs {df['Revenue'].sum():,.0f}")

Total realized revenue: Rs 30,872,142


## Task 7: Room type comparison
Occupancy proxy = confirmed room nights (how many nights were sold).

In [9]:
room_summary = confirmed.groupby('Room_Type').agg(
    Avg_Room_Rate=('Room_Rate', 'mean'),
    Bookings=('Booking_ID', 'count'),
    Room_Nights=('Nights', 'sum'),
    Revenue=('Revenue', 'sum')
).sort_values('Revenue', ascending=False)
room_summary

,Avg_Room_Rate,Bookings,Room_Nights,Revenue
Room_Type,,,,
Deluxe,"5,562.86",582,2038,"11,355,991.00"
Standard,"3,517.22",832,2993,"10,512,808.00"
Suite,"9,757.14",250,922,"9,003,343.00"


## Task 8: Booking channel ranking (volume and revenue)

In [10]:
channel_summary = df.groupby('Booking_Channel').agg(
    Total_Bookings=('Booking_ID', 'count'),
    Revenue=('Revenue', 'sum')
).sort_values('Total_Bookings', ascending=False)
channel_summary

,Total_Bookings,Revenue
Booking_Channel,,
OTA,844,"12,388,776.00"
Direct Website,560,"9,834,715.00"
Travel Agent,353,"6,210,082.00"
Corporate,243,"2,438,569.00"


In [11]:
# Multi-level groupby: Room Type x Channel
pd.pivot_table(confirmed, index='Room_Type', columns='Booking_Channel',
               values='Revenue', aggfunc='sum')

Booking_Channel,Corporate,Direct Website,OTA,Travel Agent
Room_Type,,,,
Deluxe,"903,118.00","3,529,282.00","4,600,804.00","2,322,787.00"
Standard,"908,850.00","3,265,383.00","4,452,555.00","1,886,020.00"
Suite,"626,601.00","3,040,050.00","3,335,417.00","2,001,275.00"


## Task 9: Cancellation rate by channel and lead-time band
`pd.cut()` splits numbers into groups (bands).

In [12]:
# Lead-time bands (the Task 14 categories are created here)
bins = [-1, 7, 30, 90, 1000]
labels = ['Last-Minute (0-7)', 'Short-Term (8-30)', 'Medium (31-90)', 'Advance (90+)']
df['Lead_Category'] = pd.cut(df['Lead_Time_Days'], bins=bins, labels=labels)

df['Is_Canceled'] = (df['Booking_Status'] == 'Canceled').astype(int)

channel_cancel = (df.groupby('Booking_Channel')['Is_Canceled'].mean() * 100).sort_values(ascending=False)
print('Cancellation % by channel:')
print(channel_cancel.round(2))

lead_cancel = df.groupby('Lead_Category', observed=True)['Is_Canceled'].mean() * 100
print('\nCancellation % by lead-time band:')
print(lead_cancel.round(2))

Cancellation % by channel:
Booking_Channel
OTA              24.05
Travel Agent     16.71
Direct Website   10.71
Corporate         5.76
Name: Is_Canceled, dtype: float64

Cancellation % by lead-time band:
Lead_Category
Last-Minute (0-7)    9.52
Short-Term (8-30)   15.24
Medium (31-90)      16.48
Advance (90+)       23.98
Name: Is_Canceled, dtype: float64


## Task 10: Top 3 peak revenue months

In [13]:
monthly_rev = df.groupby('Arrival_Month')['Revenue'].sum()
print('Top 3 revenue months (month number):')
print(monthly_rev.nlargest(3))

Top 3 revenue months (month number):
Arrival_Month
12   3,595,280.00
8    3,532,297.00
7    3,514,850.00
Name: Revenue, dtype: float64


## Task 11: Leisure vs Business - average stay

In [14]:
df.groupby('Traveler_Type')['Nights'].mean()

,Nights
Traveler_Type,
Business,2.03
Leisure,4.58


## Task 12: Mean, median, std of room rates using NumPy

In [15]:
rates = df['Room_Rate'].to_numpy()
print(f'Mean   : {np.mean(rates):,.2f}')
print(f'Median : {np.median(rates):,.2f}')
print(f'Std Dev: {np.std(rates):,.2f}')

Mean   : 5,148.42
Median : 4,223.50
Std Dev: 2,359.63


## Task 13: Unusually high booking values (IQR method)
**IQR rule:** Q1 = 25th percentile, Q3 = 75th percentile, IQR = Q3 − Q1.
Upper limit = Q3 + 1.5 × IQR. Any value above this is an outlier.

In [16]:
vals = confirmed['Revenue'].to_numpy()
q1, q3 = np.percentile(vals, [25, 75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(f'Q1={q1:,.0f}  Q3={q3:,.0f}  IQR={iqr:,.0f}  Upper limit={upper:,.0f}')

outliers = confirmed[confirmed['Revenue'] > upper]
print('Number of high-value outlier bookings:', len(outliers))
outliers[['Booking_ID', 'Room_Type', 'Nights', 'Room_Rate', 'Revenue']].sort_values('Revenue', ascending=False).head(10)

Q1=9,309  Q3=23,900  IQR=14,590  Upper limit=45,785
Number of high-value outlier bookings: 88


,Booking_ID,Room_Type,Nights,Room_Rate,Revenue
384,BK00225,Suite,5,"21,802.00","109,010.00"
1286,BK00412,Suite,7,"12,947.00","90,629.00"
1817,BK00234,Suite,6,"13,438.00","80,628.00"
1170,BK01243,Suite,7,"11,355.00","79,485.00"
1763,BK00980,Suite,7,"11,316.00","79,212.00"
429,BK01953,Suite,7,"10,867.00","76,069.00"
464,BK01957,Suite,3,"25,140.00","75,420.00"
1933,BK00804,Deluxe,6,"12,544.00","75,264.00"
401,BK01445,Suite,7,"10,630.00","74,410.00"
1675,BK01881,Deluxe,6,"12,254.00","73,524.00"


## Task 14: Lead-time categories
The categories were already created above using `pd.cut()`. Let's see how many bookings fall in each band.

In [17]:
df['Lead_Category'].value_counts().sort_index()

,count
Lead_Category,
Last-Minute (0-7),63
Short-Term (8-30),617
Medium (31-90),1074
Advance (90+),246
